# MMGTFFF — Experiment 2 (Phase B): MAN-SF-Style Joint Architecture

### GRU + temporal attention (price) + hierarchical tweet attention (text) combined via bilinear fusion, trained end-to-end jointly. No freezing, no late concatenation.

Run top-to-bottom on a Colab **GPU** runtime. Clones both `stocknet-dataset`
and this project's `capstone` repo (cached EDGAR fundamentals, no new API
calls).

## Why this notebook exists

Phase A (`MMGTFFF_hierarchical_tweet_attention.ipynb`) built the hierarchical
tweet-attention text encoder, but evaluated it the wrong way: the encoder was
pretrained standalone, **frozen**, and its output was only **concatenated**
to a separately-run price LSTM's final hidden state. Result: it lost to a
naive tweet-count feature on every backend (best MCC +0.049 vs. Experiment
1's +0.092). The diagnosis was architectural, not a failure of the encoder
itself — price and text never interacted during training, and concatenation
is a known-weaker fusion mechanism than what the literature actually uses.

This notebook fixes both problems by building **the actual MAN-SF
architecture**:

```
        PRICE                              TWEETS
          |                                   |
         GRU                          per-tweet embedding
          |                       (USE / FinBERT / VADER)
   Temporal Attention                         |
          |                          Tweet-level Attention
          |                                   |
          |                            Daily Tweet Vector
          |                                   |
          |                                 Day GRU
          |                                   |
          |                       Temporal Attention (text)
          |                                   |
          +----------------+------------------+
                            |
                    BILINEAR FUSION
                            |
                    Stock Representation
                            |
                      Classifier Head
                            |
                        UP / DOWN
```

Critically: **the whole thing is one network, trained end-to-end on `Target`
from scratch.** No Stage A pretraining, no frozen checkpoints — the tweet
encoder's attention weights and the price encoder's attention weights are
shaped by the same gradient signal, at the same time, so they can learn to
complement each other instead of being computed in isolation and stitched
together afterward.

## What's cheap vs. what's expensive here

- **Cheap (this notebook does this)**: jointly training the small custom
  layers on top of frozen sentence embeddings — the tweet-attention layer,
  the price GRU, the bilinear fusion tensor. Tens of thousands of parameters,
  milliseconds per batch on GPU.
- **Expensive (this notebook does NOT do this)**: fine-tuning USE/FinBERT's
  own weights end-to-end. That means backpropagating through a full
  transformer for every tweet in every batch — a much bigger compute and
  data-appetite ask, not justified at this dataset's ~20K-sample scale.

USE/FinBERT/VADER embeddings are computed once per tweet (cached to disk,
same as Phase A) and used as **fixed inputs** to the jointly-trained encoder
on top of them — never fine-tuned themselves.

## Scope

- **No GAT / graph yet.** This is MAN-SF's per-stock branch (the part before
  the graph). GAT is a separate, later step per the project roadmap.
- **No event/global tweets** — company tweets only, same as Phase A.
- **Fundamentals are a project-specific extension, not part of real MAN-SF**
  — included as a secondary ablation (concatenated after fusion), clearly
  separated from the core price+text result so the literature comparison
  stays clean.
- Runs all 3 embedding backends (USE/FinBERT/VADER) through the joint
  architecture, since backend choice is still an open question after Phase
  A's inconclusive/failed comparison.

## Grid

| Model | Contains |
|---|---|
| Price-only (GRU+attention) | sanity baseline — should land near MAN-SF's own reported "LSTM+price ~0.002 MCC" |
| Price + Fundamentals | price encoder + flattened fundamentals, concatenated |
| **Price + Text (bilinear fusion)** × 3 backends | **the actual MAN-SF-style result** — target ~0.195 MCC per the full MAN-SF paper (that number includes GAT too, which isn't here yet, so don't expect to hit it exactly) |
| Price + Text (bilinear fusion) + Fundamentals × 3 backends | extension beyond real MAN-SF |

8 models total, each trained once (no LR/MLP/LSTM trio here — once the text
encoder needs gradients flowing through it, "flatten and hand to sklearn"
no longer applies; FS1/FS2 don't have that constraint but are kept as
single GRU+attention models for a fair apples-to-apples comparison against
the fused models, rather than reusing Experiment 1's separately-tuned
LR/LSTM/MLP baselines).

## 1. Setup & Clone

In [ ]:
!git clone https://github.com/yumoxu/stocknet-dataset.git
!git clone https://github.com/AdityaMelkote3004/capstone.git
print('Cloned!')

In [ ]:
!pip install -q tensorflow-hub vaderSentiment
print('Installed.')

In [ ]:
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

STOCKNET_ROOT = 'stocknet-dataset'
CAPSTONE_ROOT = 'capstone'
PRICE_DIR = os.path.join(STOCKNET_ROOT, 'price', 'preprocessed')
TWEET_DIR = os.path.join(STOCKNET_ROOT, 'tweet', 'preprocessed')

MAX_N_DAYS = 5
MAX_INPUT_DAYS = MAX_N_DAYS - 1   # 4
MAX_N_MSGS = 30                    # StockNet config.yml max_n_msgs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('WARNING: no GPU detected -- USE/FinBERT embedding will be slow. '
          'Runtime -> Change runtime type -> GPU.')

## 2. Ticker Universe, Split Dates, Loaders

Identical to Experiment 1 and Phase A.

In [ ]:
STOCK_SYMBOLS = (
    ['XOM','RDS-B','PTR','CVX','TOT','BP','BHP','SNP','SLB','BBL'] +
    ['AAPL','PG','BUD','KO','PM','TM','PEP','UN','UL','MO'] +
    ['JNJ','PFE','NVS','UNH','MRK','AMGN','MDT','ABBV','SNY','CELG'] +
    ['AMZN','BABA','WMT','CMCSA','HD','DIS','MCD','CHTR','UPS','PCLN'] +
    ['NEE','DUK','D','SO','NGG','AEP','PCG','EXC','SRE','PPL'] +
    ['IEP','HRG','CODI','REX','SPLP','PICO','AGFS','GMRE'] +
    ['BCH','BSAC','BRK-A','JPM','WFC','BAC','V','C','HSBC','MA'] +
    ['GE','MMM','BA','HON','UTX','LMT','CAT','GD','DHR','ABB'] +
    ['GOOG','MSFT','FB','T','CHL','ORCL','TSM','VZ','INTC','CSCO']
)
assert len(STOCK_SYMBOLS) == 88

SPLIT_DATES = {
    'train': ('2014-01-01', '2015-08-01'),
    'dev':   ('2015-08-01', '2015-10-01'),
    'test':  ('2015-10-01', '2016-01-01'),
}

def get_split(date):
    ds = date.isoformat()
    for name, (start, end) in SPLIT_DATES.items():
        if start <= ds < end:
            return name
    return None


def load_movement_file(ticker):
    fp = os.path.join(PRICE_DIR, f'{ticker}.txt')
    if not os.path.exists(fp):
        return None
    rows = []
    with open(fp, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            date = datetime.strptime(parts[0], '%Y-%m-%d').date()
            mv = float(parts[1])
            prices = [float(parts[3]), float(parts[4]), float(parts[5])]
            rows.append((date, mv, prices))
    rows.sort(key=lambda r: r[0])
    return rows


def load_tweet_dates(ticker):
    tdir = os.path.join(TWEET_DIR, ticker)
    out = {}
    if not os.path.isdir(tdir):
        return out
    for fname in os.listdir(tdir):
        try:
            d = datetime.strptime(fname, '%Y-%m-%d').date()
        except ValueError:
            continue
        msgs = []
        try:
            with open(os.path.join(tdir, fname), 'r', encoding='utf8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    obj = json.loads(line)
                    text = obj.get('text', '')
                    if isinstance(text, list) and text:
                        msgs.append(text)
        except Exception:
            continue
        if msgs:
            out[d] = msgs
    return out

print('Loaders defined.')

## 3. Build Samples — Same Windowing, Buffer-Zone Filter, and Tweet Alignment as Before

Unchanged from Phase A: calendar-anchored window, buffer-zone filter applied
only to the target day, and every calendar day's tweets attach to the first
actual trading day strictly after it.

In [ ]:
def build_samples_with_tweet_refs(ticker, movement_rows, tweet_by_date, max_n_days=MAX_N_DAYS):
    date_list = [r[0] for r in movement_rows]
    mv_by_date = {r[0]: r[1] for r in movement_rows}
    prices_by_date = {r[0]: r[2] for r in movement_rows}
    date_set = set(date_list)
    max_input_days = max_n_days - 1

    samples = []
    for main_target_date in date_list:
        main_mv = mv_by_date[main_target_date]
        if -0.005 <= main_mv < 0.0055:
            continue

        d_t_min = main_target_date - timedelta(days=max_n_days - 1)
        ts_window = sorted(d for d in date_set if d_t_min <= d < main_target_date)
        T_input = len(ts_window)
        if T_input == 0:
            continue

        ts_full = ts_window + [main_target_date]

        d_d_max = main_target_date - timedelta(days=1)
        d_d_min = main_target_date - timedelta(days=max_n_days)
        unaligned_days = sorted(d for d in tweet_by_date if d_d_min <= d <= d_d_max)

        slot_refs = [[] for _ in range(T_input + 1)]
        for d in unaligned_days:
            n_msgs = len(tweet_by_date[d])
            for t in range(T_input + 1):
                if d < ts_full[t]:
                    slot_refs[t].extend((d, i) for i in range(n_msgs))
                    break

        input_prices = [prices_by_date[d] for d in ts_window]
        input_prices += [[0.0, 0.0, 0.0]] * (max_input_days - T_input)

        input_tweet_refs = [refs[:MAX_N_MSGS] for refs in slot_refs[:-1]]
        input_tweet_refs += [[]] * (max_input_days - T_input)

        samples.append({
            'Ticker': ticker,
            'Target_Date': main_target_date,
            'Target': 1 if main_mv > 0 else 0,
            'Window_Length': T_input,
            'Input_Prices': input_prices,
            'Input_Tweet_Refs': input_tweet_refs,
        })

    return samples


all_samples = []
tweet_by_date_all = {}
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        continue
    tweet_by_date = load_tweet_dates(ticker)
    tweet_by_date_all[ticker] = tweet_by_date
    samples = build_samples_with_tweet_refs(ticker, movement_rows, tweet_by_date)
    for s in samples:
        s['Split'] = get_split(s['Target_Date'])
    all_samples.extend(s for s in samples if s['Split'] is not None)

print(f'Total samples: {len(all_samples)}')
assert 25000 < len(all_samples) < 28000, \
    f'Sample count {len(all_samples)} unexpected -- check windowing logic.'
print('Sample count matches the expected StockNet dataset scale.')

## 4. Fundamentals (Secondary Ablation Only) — Reused, Unchanged

In [ ]:
fund_raw = pd.read_csv(os.path.join(CAPSTONE_ROOT, 'dataset', 'final', 'edgar_raw_fundamentals.csv'))
fund_raw['Filed_Date'] = pd.to_datetime(fund_raw['Filed_Date']).dt.date
FUNDAMENTAL_FEATURES = ['Revenue', 'NetIncome', 'TotalAssets', 'TotalLiabilities',
                         'StockholdersEquity', 'EPS', 'Cash']
N_FUND = len(FUNDAMENTAL_FEATURES) + 1  # +ROA
FUND_WINDOW_DIM = N_FUND * MAX_INPUT_DAYS  # flattened, same convention as Experiment 1's FS4

def align_fundamentals_for_ticker(ticker, all_dates):
    daily_index = pd.DatetimeIndex(pd.to_datetime(all_dates))
    ticker_data = fund_raw[fund_raw['Ticker'] == ticker]
    result = {}
    if len(ticker_data) == 0:
        for m in FUNDAMENTAL_FEATURES:
            result[m] = pd.Series(np.nan, index=daily_index)
        result['ROA'] = pd.Series(np.nan, index=daily_index)
        return result
    for metric in FUNDAMENTAL_FEATURES:
        md_ = (ticker_data[ticker_data['Metric'] == metric]
               .sort_values('Filed_Date').drop_duplicates(subset=['Filed_Date'], keep='last'))
        if len(md_) == 0:
            result[metric] = pd.Series(np.nan, index=daily_index)
            continue
        ts = md_.set_index(pd.to_datetime(md_['Filed_Date']))['Value']
        ts = ts[~ts.index.duplicated(keep='last')]
        result[metric] = ts.reindex(daily_index, method='ffill')
    roa = result['NetIncome'] / result['TotalAssets']
    result['ROA'] = roa.replace([np.inf, -np.inf], np.nan)
    return result


fund_lookup_by_ticker = {}
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        continue
    all_dates = [r[0] for r in movement_rows]
    fund_series = align_fundamentals_for_ticker(ticker, all_dates)
    fund_lookup_by_ticker[ticker] = {
        d: [fund_series[m].iloc[i] for m in FUNDAMENTAL_FEATURES + ['ROA']]
        for i, d in enumerate(all_dates)
    }

for s in all_samples:
    ticker, target = s['Ticker'], s['Target_Date']
    lookup = fund_lookup_by_ticker.get(ticker, {})
    d_t_min = target - timedelta(days=MAX_N_DAYS - 1)
    ts_window = sorted(d for d in lookup if d_t_min <= d < target)
    fund_window = [lookup[d] for d in ts_window]
    fund_window += [[np.nan] * N_FUND] * (MAX_INPUT_DAYS - len(ts_window))
    s['Input_Fundamentals_Flat'] = np.nan_to_num(
        np.array(fund_window, dtype=np.float32), nan=0.0
    ).flatten()  # (MAX_INPUT_DAYS * N_FUND,)

print('Fundamentals aligned.')

## 5. Per-Tweet Embedding Functions (Cached to Disk)

Same as Phase A: each backend embeds every unique tweet once, model is
loaded/discarded per backend, and results cache to disk so a crash/restart
doesn't force recomputation.

In [ ]:
def flatten_all_tweets(tweet_by_date_all):
    flat_keys, flat_texts = [], []
    for ticker, by_date in tweet_by_date_all.items():
        for date, msgs in by_date.items():
            for i, tokens in enumerate(msgs):
                flat_keys.append((ticker, date, i))
                flat_texts.append(' '.join(tokens))
    return flat_keys, flat_texts


flat_keys, flat_texts = flatten_all_tweets(tweet_by_date_all)
key_to_row_global = {k: i for i, k in enumerate(flat_keys)}
print(f'Unique tweets to embed: {len(flat_texts)}')


def embed_use(texts, batch_size=512):
    import tensorflow as tf
    import tensorflow_hub as hub
    model = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')
    out = np.zeros((len(texts), 512), dtype=np.float32)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        out[start:start + len(batch)] = model(batch).numpy()
        if start % (batch_size * 20) == 0:
            print(f'  USE: {start}/{len(texts)}')
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return out


def embed_finbert(texts, batch_size=64, max_length=64):
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModel.from_pretrained('ProsusAI/finbert').to(DEVICE).eval()
    out = np.zeros((len(texts), 768), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            inputs = tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            hidden = model(**inputs).last_hidden_state[:, 0, :]
            out[start:start + len(batch)] = hidden.cpu().numpy()
            if start % (batch_size * 50) == 0:
                print(f'  FinBERT: {start}/{len(texts)}')
    del model, tok
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return out


def embed_vader(texts):
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()
    out = np.zeros((len(texts), 4), dtype=np.float32)
    for i, text in enumerate(texts):
        s = analyzer.polarity_scores(text)
        out[i] = [s['neg'], s['neu'], s['pos'], s['compound']]
    return out


BACKEND_CONFIGS = {
    'USE':     (embed_use, 512),
    'FinBERT': (embed_finbert, 768),
    'VADER':   (embed_vader, 4),
}

CACHE_DIR = 'final/tweet_embeddings_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def get_backend_embeddings(backend_name):
    embed_fn, emb_dim = BACKEND_CONFIGS[backend_name]
    cache_path = os.path.join(CACHE_DIR, f'{backend_name}_embeddings.npy')
    if os.path.exists(cache_path):
        print(f'{backend_name}: loading cached embeddings from {cache_path}')
        return np.load(cache_path), emb_dim
    print(f"\n{'='*50}\n{backend_name}: embedding {len(flat_texts)} tweets...\n{'='*50}")
    embeddings = embed_fn(flat_texts)
    np.save(cache_path, embeddings)
    print(f'{backend_name}: cached to {cache_path}')
    return embeddings, emb_dim

print('Embedding functions defined.')

## 6. The MAN-SF Joint Architecture

Three pieces, all trained together as one network:

1. **`PriceEncoder`**: GRU over the (masked) 4-day price window + temporal
   attention — structurally identical to the tweet encoder's day-level GRU +
   temporal attention below, just applied directly to price features instead
   of pre-pooled daily tweet vectors.
2. **`HierarchicalTweetEncoder`**: unchanged from Phase A — tweet-level
   attention → day-level GRU → temporal attention.
3. **`BilinearFusion`**: `nn.Bilinear` combines the price and text embeddings
   into one fused representation — literally the mechanism MAN-SF's own
   ablation table shows beating concatenation (≈0.156 MCC) and attention
   fusion (≈0.173 MCC) on this exact dataset (≈0.195 MCC).

Fundamentals, when included, are concatenated **after** fusion — they're not
part of real MAN-SF, so they don't get folded into the bilinear step.

In [ ]:
class PriceEncoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.temporal_score = nn.Linear(hidden_dim, 1)
        self.output_dim = hidden_dim

    def forward(self, x, mask):
        lengths = mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        gru_out, _ = self.gru(packed)
        gru_out, _ = nn.utils.rnn.pad_packed_sequence(gru_out, batch_first=True, total_length=x.shape[1])
        scores = self.temporal_score(gru_out).squeeze(-1).masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * gru_out).sum(dim=1)


class TweetLevelAttention(nn.Module):
    def __init__(self, input_dim, proj_dim=64):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.score = nn.Linear(proj_dim, 1)

    def forward(self, tweet_embs, tweet_mask):
        h = torch.tanh(self.proj(tweet_embs))
        scores = self.score(h).squeeze(-1)
        scores = scores.masked_fill(tweet_mask == 0, -1e9)
        has_any = (tweet_mask.sum(dim=-1, keepdim=True) > 0).float()
        weights = torch.softmax(scores, dim=-1) * has_any
        return (weights.unsqueeze(-1) * h).sum(dim=2)


class HierarchicalTweetEncoder(nn.Module):
    def __init__(self, input_dim, proj_dim=64, hidden_dim=64):
        super().__init__()
        self.tweet_attn = TweetLevelAttention(input_dim, proj_dim)
        self.gru = nn.GRU(proj_dim, hidden_dim, batch_first=True)
        self.temporal_score = nn.Linear(hidden_dim, 1)
        self.output_dim = hidden_dim

    def forward(self, tweet_embs, tweet_mask, window_mask):
        daily = self.tweet_attn(tweet_embs, tweet_mask)
        lengths = window_mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(daily, lengths, batch_first=True, enforce_sorted=False)
        gru_out, _ = self.gru(packed)
        gru_out, _ = nn.utils.rnn.pad_packed_sequence(gru_out, batch_first=True, total_length=daily.shape[1])
        scores = self.temporal_score(gru_out).squeeze(-1).masked_fill(window_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * gru_out).sum(dim=1)


class BilinearFusion(nn.Module):
    def __init__(self, price_dim=64, text_dim=64, fusion_dim=64):
        super().__init__()
        self.bilinear = nn.Bilinear(price_dim, text_dim, fusion_dim)

    def forward(self, price_emb, text_emb):
        return torch.tanh(self.bilinear(price_emb, text_emb))


class MANSFModel(nn.Module):
    """price-only (text_input_dim=None) / price+fundamentals / price+text
    (bilinear fusion) / price+text+fundamentals -- one class covers all 4
    grid entries via optional components.

    FIX (residual path around the bilinear branch): an earlier version fed
    the classifier head ONLY self.fusion(price_repr, text_repr) when text was
    present. That collapsed to predicting a single constant class for every
    sample, identically across USE, FinBERT, and VADER (bit-for-bit identical
    accuracy/F1 across backends -- proof it was an optimization pathology in
    the bilinear branch, not an embedding-quality problem, since three
    different embeddings produced the exact same degenerate output). The
    fix concatenates price_repr -- already shown to train to a real,
    non-degenerate MCC on its own -- alongside the fusion output, so the
    head always has a working signal to fall back on even if the bilinear
    branch struggles early in training, instead of routing everything
    through it exclusively."""
    def __init__(self, price_input_dim=3, text_input_dim=None, hidden_dim=64,
                 fusion_dim=64, fund_dim=0, dropout=0.2):
        super().__init__()
        self.price_encoder = PriceEncoder(price_input_dim, hidden_dim)
        self.use_text = text_input_dim is not None
        if self.use_text:
            self.text_encoder = HierarchicalTweetEncoder(text_input_dim, hidden_dim, hidden_dim)
            self.fusion = BilinearFusion(hidden_dim, hidden_dim, fusion_dim)
            rep_dim = fusion_dim + hidden_dim  # fusion output + price_repr residual
        else:
            rep_dim = hidden_dim
        self.fund_dim = fund_dim
        head_in = rep_dim + fund_dim
        self.head = nn.Sequential(
            nn.Linear(head_in, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 2))

    def forward(self, price_x, window_mask, tweet_emb=None, tweet_mask=None, fund_x=None):
        price_repr = self.price_encoder(price_x, window_mask)
        if self.use_text:
            text_repr = self.text_encoder(tweet_emb, tweet_mask, window_mask)
            fused = self.fusion(price_repr, text_repr)
            rep = torch.cat([fused, price_repr], dim=-1)
        else:
            rep = price_repr
        if self.fund_dim > 0:
            rep = torch.cat([rep, fund_x], dim=-1)
        return self.head(rep)


def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.5,
    }

print('MAN-SF architecture defined.')

## 7. Joint Training Dataset and Loop

`JointDataset` builds each sample's tweet window tensor lazily from a flat
`(N, dim)` embeddings array (same memory-safety fix as Phase A — never
materializes dense per-sample tensors for the whole dataset). Price and
fundamentals are small enough to store directly per sample.

In [ ]:
class JointDataset(Dataset):
    def __init__(self, samples, embeddings=None, key_to_row=None, emb_dim=0, use_fund=False):
        self.samples = samples
        self.embeddings = embeddings
        self.key_to_row = key_to_row
        self.emb_dim = emb_dim
        self.use_fund = use_fund
        self.use_text = embeddings is not None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        price_x = np.array(s['Input_Prices'], dtype=np.float32)
        wmask = np.zeros(MAX_INPUT_DAYS, dtype=np.float32)
        wmask[:s['Window_Length']] = 1.0
        y = s['Target']

        if self.use_text:
            te = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS, self.emb_dim), dtype=np.float32)
            tm = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS), dtype=np.float32)
            ticker = s['Ticker']
            for day_i, refs in enumerate(s['Input_Tweet_Refs']):
                for msg_i, (date, idx) in enumerate(refs[:MAX_N_MSGS]):
                    row = self.key_to_row.get((ticker, date, idx))
                    if row is not None:
                        te[day_i, msg_i] = self.embeddings[row]
                        tm[day_i, msg_i] = 1.0
        else:
            te = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS, 1), dtype=np.float32)
            tm = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS), dtype=np.float32)

        fund_x = s['Input_Fundamentals_Flat'] if self.use_fund else np.zeros(0, dtype=np.float32)

        return (torch.tensor(price_x), torch.tensor(wmask), torch.tensor(te), torch.tensor(tm),
                torch.tensor(fund_x), torch.tensor(y, dtype=torch.long))


def normalize_price(train_samples, *other_sample_lists):
    train_prices = np.array([s['Input_Prices'] for s in train_samples], dtype=np.float32)
    flat = train_prices.reshape(-1, 3)
    mean, std = flat.mean(axis=0), flat.std(axis=0)
    std[std == 0] = 1.0
    def apply(samples):
        for s in samples:
            s['Input_Prices'] = ((np.array(s['Input_Prices'], dtype=np.float32) - mean) / std).tolist()
    apply(train_samples)
    for lst in other_sample_lists:
        apply(lst)


def normalize_fund(train_samples, *other_sample_lists):
    train_fund = np.array([s['Input_Fundamentals_Flat'] for s in train_samples], dtype=np.float32)
    mean, std = train_fund.mean(axis=0), train_fund.std(axis=0)
    std[std == 0] = 1.0
    def apply(samples):
        for s in samples:
            s['Input_Fundamentals_Flat'] = (s['Input_Fundamentals_Flat'] - mean) / std
    apply(train_samples)
    for lst in other_sample_lists:
        apply(lst)


def train_mansf(train_samples, val_samples, test_samples, text_input_dim=None,
                 embeddings=None, key_to_row=None, use_fund=False,
                 epochs=50, patience=10):
    torch.manual_seed(SEED)
    fund_dim = FUND_WINDOW_DIM if use_fund else 0
    model = MANSFModel(price_input_dim=3, text_input_dim=text_input_dim,
                        fund_dim=fund_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    kwargs = dict(embeddings=embeddings, key_to_row=key_to_row,
                  emb_dim=(text_input_dim or 0), use_fund=use_fund)
    train_dl = DataLoader(JointDataset(train_samples, **kwargs), batch_size=64, shuffle=True)
    val_ds = JointDataset(val_samples, **kwargs)
    test_ds = JointDataset(test_samples, **kwargs)

    def evaluate(ds):
        model.eval()
        dl = DataLoader(ds, batch_size=256)
        yt, yp, ypr = [], [], []
        with torch.no_grad():
            for price_x, wmask, te, tm, fund_x, y in dl:
                price_x, wmask = price_x.to(DEVICE), wmask.to(DEVICE)
                te, tm, fund_x = te.to(DEVICE), tm.to(DEVICE), fund_x.to(DEVICE)
                logits = model(price_x, wmask, te if text_input_dim else None,
                               tm if text_input_dim else None, fund_x if use_fund else None)
                probs = torch.softmax(logits, dim=1)[:, 1]
                yt.extend(y.numpy()); yp.extend(logits.argmax(1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
        return compute_metrics(np.array(yt), np.array(yp), np.array(ypr))

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for price_x, wmask, te, tm, fund_x, y in train_dl:
            price_x, wmask, y = price_x.to(DEVICE), wmask.to(DEVICE), y.to(DEVICE)
            te, tm, fund_x = te.to(DEVICE), tm.to(DEVICE), fund_x.to(DEVICE)
            opt.zero_grad()
            logits = model(price_x, wmask, te if text_input_dim else None,
                           tm if text_input_dim else None, fund_x if use_fund else None)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(val_ds)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return evaluate(test_ds)

print('Training loop defined.')

## 8. Run the Grid

`Price-only` and `Price+Fundamentals` don't depend on a text backend, so
they run once. `Price+Text` and `Price+Text+Fundamentals` run once per
backend (USE/FinBERT/VADER) — 8 joint models total.

In [ ]:
by_split = {'train': [], 'dev': [], 'test': []}
for s in all_samples:
    by_split[s['Split']].append(s)

normalize_price(by_split['train'], by_split['dev'], by_split['test'])
normalize_fund(by_split['train'], by_split['dev'], by_split['test'])

results = {}

print(f"\n{'='*60}\nPrice-only (GRU + temporal attention)\n{'='*60}")
m = train_mansf(by_split['train'], by_split['dev'], by_split['test'])
results['Price_Only'] = {'label': 'Price only', 'backend': None, **m}
print(f"  MCC={m['mcc']:+.4f} F1={m['f1']:.4f} Acc={m['accuracy']:.4f} AUC={m['auc']:.4f}")

print(f"\n{'='*60}\nPrice + Fundamentals\n{'='*60}")
m = train_mansf(by_split['train'], by_split['dev'], by_split['test'], use_fund=True)
results['Price_Fundamentals'] = {'label': 'Price + Fundamentals', 'backend': None, **m}
print(f"  MCC={m['mcc']:+.4f} F1={m['f1']:.4f} Acc={m['accuracy']:.4f} AUC={m['auc']:.4f}")

for backend_name in BACKEND_CONFIGS:
    embeddings, emb_dim = get_backend_embeddings(backend_name)

    print(f"\n{'='*60}\nPrice + Text (bilinear fusion) [{backend_name}]\n{'='*60}")
    m = train_mansf(by_split['train'], by_split['dev'], by_split['test'],
                     text_input_dim=emb_dim, embeddings=embeddings, key_to_row=key_to_row_global)
    results[f'Price_Text__{backend_name}'] = {'label': 'Price + Text (bilinear)', 'backend': backend_name, **m}
    print(f"  MCC={m['mcc']:+.4f} F1={m['f1']:.4f} Acc={m['accuracy']:.4f} AUC={m['auc']:.4f}")

    print(f"\n{'='*60}\nPrice + Text (bilinear fusion) + Fundamentals [{backend_name}]\n{'='*60}")
    m = train_mansf(by_split['train'], by_split['dev'], by_split['test'],
                     text_input_dim=emb_dim, embeddings=embeddings, key_to_row=key_to_row_global,
                     use_fund=True)
    results[f'Price_Text_Fundamentals__{backend_name}'] = {
        'label': 'Price + Text (bilinear) + Fundamentals', 'backend': backend_name, **m}
    print(f"  MCC={m['mcc']:+.4f} F1={m['f1']:.4f} Acc={m['accuracy']:.4f} AUC={m['auc']:.4f}")

    del embeddings
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

print('\nAll 8 joint models trained.')

## 9. Summary Table and Save

In [ ]:
rows = []
for key, res in results.items():
    rows.append({
        'Model': res['label'], 'Backend': res['backend'] or '-',
        'Accuracy': res['accuracy'], 'F1': res['f1'], 'MCC': res['mcc'], 'AUC': res['auc'],
    })
results_df = pd.DataFrame(rows).sort_values('MCC', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

best_row = results_df.iloc[0]
print(f"\nBest overall: {best_row['Model']} [{best_row['Backend']}] -- MCC={best_row['MCC']:+.4f}")

os.makedirs('final', exist_ok=True)
with open('final/mansf_results.json', 'w') as f:
    json.dump(results, f, indent=2)
results_df.to_csv('final/mansf_summary.csv', index=False)
print('\nSaved: final/mansf_results.json, final/mansf_summary.csv')

## 10. Compare Against Prior Experiments

| | MCC |
|---|---:|
| Experiment 1 best (LSTM, raw tweet count) | +0.092 |
| Experiment 2 Phase A best (frozen + late concat, any backend) | +0.080 |
| MAN-SF paper: LSTM + price only | ~0.002 |
| MAN-SF paper: GRU + social text | ~0.077 |
| MAN-SF paper: bilinear fusion (full architecture incl. GAT) | ~0.195 |
| **This notebook: Price-only (GRU+attention)** | *(see Section 9)* |
| **This notebook: Price+Text (bilinear fusion), best backend** | *(see Section 9)* |

If Price+Text (bilinear) here clearly beats both Experiment 1 and Phase A,
that confirms the diagnosis from Phase A was right — the *fusion mechanism*
was the problem, not the tweet encoder. Don't expect to hit MAN-SF's full
~0.195 exactly: that number includes GAT, which isn't in this notebook yet —
GAT is the next roadmap step once this per-stock branch is validated.

## Fixed: class-collapse in the bilinear branch (confirmed on real Colab run)

An earlier version of this notebook fed the classifier head **only**
`self.fusion(price_repr, text_repr)` when text was included. Running it on
Colab with real USE/FinBERT embeddings confirmed a systematic collapse:
Price+Text (bilinear) came back with **MCC exactly 0.0 for both USE and
FinBERT**, with bit-for-bit identical accuracy and F1 to 16 decimal places
across the two backends (and VADER landed on the same pattern, off by 1-2
samples). Identical hard predictions regardless of which embedding was fed
in is proof this was an optimization pathology in the bilinear branch
itself, not an embedding-quality problem -- three different embeddings
cannot coincidentally produce bit-identical wrong answers unless the model
learned to ignore the text branch entirely and output a constant class.

Confirming evidence: the collapse disappeared the moment fundamentals were
concatenated after fusion (MCC +0.041 USE, +0.060 FinBERT, +0.041 VADER) --
because the head then had *some* real per-sample signal to learn from even
if the bilinear branch was degenerate. Price-only and Price+Fundamentals
never touch `BilinearFusion` and were never at risk, consistent with them
being the only two non-degenerate results in the original run.

**The fix** (already applied in `MANSFModel` above): concatenate
`price_repr` -- already shown to train to a real, non-degenerate MCC on its
own -- alongside the bilinear fusion output, instead of routing everything
through the bilinear branch exclusively. This gives the classifier head a
working fallback signal so it can't fully collapse even if the bilinear
branch struggles early in training.

**Still worth checking when you rerun this**: verify Price+Text (bilinear)
no longer shows MCC exactly 0.0, and that USE/FinBERT/VADER no longer
produce identical hard predictions. If the collapse somehow persists after
this fix, that would point to a more fundamental optimization issue (try a
different seed, lower learning rate, or drop the `tanh` in `BilinearFusion`
next) rather than something to accept as normal.